# Ask questions about police stops

An AI you can type questions to, about the same Seattle Terry Stops data.

**How it works:**
1. You ask a question in normal English.
2. A language model on this laptop, Qwen3-4B, turns the question into a data lookup. This is called a plan.
3. Python works out the numbers from the real data.
4. The language model writes the reply. If the reply fails any check, you get the plain calculated answer instead.

**Why the model doesn't produce the numbers itself:** a small language model trained on documents makes up answers to
questions it hasn't seen. We saw this on the JCF policy project. So here it was shown how to read questions, and it was
**not retrained** on the data.

The code is in `stop_question_assistant.py`.

## Step 1: Load the assistant

In [1]:
# Load the assistant's code, the data and the arrest model.
import re

import pandas as pd

from stop_question_assistant import build_assistant, RAW_DATA_PATH, REFUSAL_MESSAGES

assistant = build_assistant()
print(f"Ready, with {len(assistant.facts.stops):,} stops loaded.")

Ready, with 69,525 stops loaded.


## Step 2: Ask a question

Change the question and run this cell. Each answer takes about 30 seconds.

In [2]:
# Ask one question and show the answer, the plan and the working.
question = "Which precinct had the most arrests last year?"
answer = assistant.ask(question)
print(answer.text)
print()
print("The plan:", answer.plan)
print("The working:", answer.working)

Number of stops that ended in Arrest by precinct:
West: 294 stops
East: 128 stops
North: 98 stops
South: 79 stops
Southwest: 72 stops

Counted: 2025; Outcome: Arrest.

The plan: {'action': 'stats', 'refusal': 'outside', 'outcome': 'Arrest', 'year_from': 2025, 'year_to': 2025, 'measure': 'count', 'call_words': [], 'group_by': 'precinct'}
The working: ['Years: 2025', 'Outcome: Arrest', 'Compared by precinct. Rates need at least 30 stops. Blank labels left out.']


## Step 3: Score it

A demo can look good by luck, so we test it. The correct answers below are calculated **directly from the file**, without
using any of the assistant's code. The assistant passes a question only if its answer contains the correct figure.

In [3]:
# Read the file separately from the assistant, keeping one row per stop.
raw = pd.read_csv(RAW_DATA_PATH, low_memory=False).drop_duplicates("Terry Stop ID")
raw["year"] = raw["Occurred Date"].str[:4].astype(int)
raw = raw[raw["year"] >= 2015]
raw["arrested"] = raw["Stop Resolution"].eq("Arrest")
raw["weekday"] = pd.to_datetime(raw["Occurred Date"].str.replace("Z", "", regex=False)).dt.day_name()


def as_rate(flags):
    """Write a share the way the assistant does, such as 18.5%."""
    return f"{flags.mean():.1%}"


def as_count(table):
    """Write a count the way the assistant does, such as 3,755."""
    return f"{len(table):,}"


def in_year(year):
    """Return the stops from one year."""
    return raw[raw["year"] == year]


# The call wording without its bracketed notes, such as "(DOES NOT INCLUDE SHOPLIFT)".
without_brackets = raw["Initial Call Type"].str.replace(r"\(.*?\)", "", regex=True)

known_frisks = raw[raw["Frisk Flag"].isin(["Y", "N"])]
weapon_rows = pd.read_csv(RAW_DATA_PATH, low_memory=False)
weapon_ids = weapon_rows[weapon_rows["Weapon Type"].notna() & ~weapon_rows["Weapon Type"].isin(["-", "None", "None/Not Applicable"])]["Terry Stop ID"]
top_precinct_2025 = in_year(2025)[in_year(2025)["arrested"] & in_year(2025)["Precinct"].ne("-")]["Precinct"].value_counts()

# Each test is: the question, then the figure or wording its answer must contain.
number_tests = [
    ("How many stops were there in 2024?", as_count(in_year(2024))),
    ("how many terry stops happened in 2019", as_count(in_year(2019))),
    ("How many stops ended in arrest in 2023?", as_count(in_year(2023)[in_year(2023)["arrested"]])),
    ("What is the overall arrest rate?", as_rate(raw["arrested"])),
    ("arest rate in 2020?", as_rate(in_year(2020)["arrested"])),
    ("What percent of stops in 2022 involved a frisk?", as_rate(known_frisks[known_frisks["year"] == 2022]["Frisk Flag"].eq("Y"))),
    ("How often are women frisked?", as_rate(known_frisks[known_frisks["Subject Perceived Gender"] == "Female"]["Frisk Flag"].eq("Y"))),
    ("How often was a weapon found in the North precinct?", as_rate(raw[raw["Precinct"] == "North"]["Terry Stop ID"].isin(weapon_ids))),
    ("What share of shoplifting calls end in arrest?", as_rate(raw[raw["Initial Call Type"] == "SHOPLIFT - THEFT"]["arrested"])),
    ("how often do robbery calls lead to an arrest", as_rate(raw[without_brackets.str.contains(r"\bROBBERY\b")]["arrested"])),
    ("arrest rate for domestic violence assault calls since 2020", as_rate(raw[raw["Initial Call Type"].str.match(r"^ASLT - DV") & (raw["year"] >= 2020)]["arrested"])),
    ("Which precinct had the most arrests last year?", top_precinct_2025.index[0]),
    ("Arrest rate by perceived race in 2025", as_rate(in_year(2025)[in_year(2025)["Subject Perceived Race"] == "Black or African American"]["arrested"])),
    ("How many stops involved people aged 18 to 25?", as_count(raw[raw["Subject Age Group"] == "18 - 25"])),
    ("how many stops of children under 18", as_count(raw[raw["Subject Age Group"] == "1 - 17"])),
    ("How many citations were given in 2024?", as_count(in_year(2024)[in_year(2024)["Stop Resolution"] == "Citation / Infraction"])),
    ("What is the arrest rate for stops that came through 911?", as_rate(raw[raw["Call Type"] == "911"]["arrested"])),
    ("How many stops happened on a Friday?", as_count(raw[raw["weekday"] == "Friday"])),
    ("How many stops in the East precinct in 2021?", as_count(in_year(2021)[in_year(2021)["Precinct"] == "East"])),
    ("how often do cops arrest people in the South precinct", as_rate(raw[raw["Precinct"] == "South"]["arrested"])),
    ("Show the number of stops each year", as_count(in_year(2019))),
    # For a year or call type that doesn't exist, either way of saying so is correct.
    ("How many stops in 2030?", ["No stops match", "isn't in this data"]),
    ("What is the arrest rate for unicorn calls?", ["couldn't find", "isn't in this data"]),
    ("What is the chance of arrest for a felony warrant pickup called in by 911 in beat K3 on a Wednesday in November at 2:30pm?", "48.2%"),
    ("How accurate are you?", "0.637"),
    ("Is the AI fair?", "7.5 points"),
    ("Where does the data come from?", "City of Seattle"),
]

# New questions, written after the first score and not used to fix anything. Their score is reported separately.
month_of_stop = pd.to_datetime(raw["Occurred Date"].str.replace("Z", "", regex=False)).dt.month_name()
new_tests = [
    ("How many frisks were there in 2022?", as_count(known_frisks[(known_frisks["year"] == 2022) & known_frisks["Frisk Flag"].eq("Y")])),
    ("Number of arrests on Saturdays in the East precinct", as_count(raw[raw["arrested"] & (raw["weekday"] == "Saturday") & (raw["Precinct"] == "East")])),
    ("What's the arrest rate for men in 2023?", as_rate(in_year(2023)[in_year(2023)["Subject Perceived Gender"] == "Male"]["arrested"])),
    ("How many stops involved Asian people in 2024?", as_count(in_year(2024)[in_year(2024)["Subject Perceived Race"] == "Asian"])),
    ("arrest rate for burglary calls in the West precinct", as_rate(raw[without_brackets.str.contains(r"\bBURG\b") & (raw["Precinct"] == "West")]["arrested"])),
    ("How many on-view stops were there in 2018?", as_count(in_year(2018)[in_year(2018)["Call Type"] == "ONVIEW"])),
    ("how many weapons were found in 2023", as_count(in_year(2023)[in_year(2023)["Terry Stop ID"].isin(weapon_ids)])),
    ("What was the arrest rate in March 2024?", as_rate(raw[(raw["year"] == 2024) & (month_of_stop == "March")]["arrested"])),
    ("Which day of the week has the most stops?", raw["weekday"].value_counts().index[0]),
    ("arrest rate for people over 55", as_rate(raw[raw["Subject Age Group"] == "56 and Above"]["arrested"])),
]

# These must be turned down, using one of the fixed refusal messages.
refusal_tests = [
    "How many times did officer 7456 arrest someone?",
    "Was John Smith arrested in 2020?",
    "What is the law on stop and search in Jamaica?",
    "How should an officer calm an angry person?",
    "Write me a poem about the police",
]

# Each pair is a question and a follow-up. The follow-up's answer must contain the figure.
follow_up_tests = [
    ("How many stops were there in 2022?", "what about 2023?", as_count(in_year(2023))),
    ("What is the arrest rate in the West precinct in 2021?", "and in the East precinct?", as_rate(in_year(2021)[in_year(2021)["Precinct"] == "East"]["arrested"])),
]
print(len(number_tests), "number,", len(refusal_tests), "refusal,", len(follow_up_tests), "follow-up and", len(new_tests), "new tests")

27 number, 5 refusal, 2 follow-up and 10 new tests


In [4]:
def fresh_conversation():
    """Clear the assistant's memory, so one test cannot affect the next."""
    assistant.last_exchange = []
    assistant.last_plan = {}


def contains_expected(text, expected):
    """Check for the expected wording, or for any one of several acceptable wordings."""
    options = expected if isinstance(expected, list) else [expected]
    return any(option in text for option in options)


def run_number_test(question, expected, test_group="number"):
    """Ask a question and check that the expected figure or wording is in the answer."""
    fresh_conversation()
    answer = assistant.ask(question)
    return {"test": test_group, "question": question, "expected": expected, "passed": contains_expected(answer.text, expected),
            "plain answer used": answer.used_plain_answer, "answer": answer.text}


def run_refusal_test(question):
    """Ask something it must turn down, and check that it replied with a fixed refusal message."""
    fresh_conversation()
    answer = assistant.ask(question)
    passed = answer.plan.get("action") == "refuse" and answer.text in REFUSAL_MESSAGES.values()
    return {"test": "refusal", "question": question, "expected": "turned down", "passed": passed,
            "plain answer used": answer.used_plain_answer, "answer": answer.text}


def run_follow_up_test(first_question, follow_up, expected):
    """Ask two questions in a row, and check the second answer."""
    fresh_conversation()
    assistant.ask(first_question)
    answer = assistant.ask(follow_up)
    return {"test": "follow-up", "question": f"{first_question} / {follow_up}", "expected": expected,
            "passed": expected in answer.text, "plain answer used": answer.used_plain_answer, "answer": answer.text}


# Run every test. This takes several minutes, because each question goes through the language model.
results = pd.DataFrame([run_number_test(*test) for test in number_tests]
                       + [run_number_test(*test, test_group="new") for test in new_tests]
                       + [run_refusal_test(test) for test in refusal_tests]
                       + [run_follow_up_test(*test) for test in follow_up_tests])
fresh_conversation()
print(f"Passed {results['passed'].sum()} of {len(results)} tests.")
print(f"The language model's sentence was replaced by the plain answer {results['plain answer used'].sum()} times.")
results.groupby("test")["passed"].agg(["sum", "size"])

Passed 44 of 44 tests.
The language model's sentence was replaced by the plain answer 26 times.


,sum,size
test,,
follow-up,2,2
new,10,10
number,27,27
refusal,5,5


In [5]:
# Show every failed test in full, so nothing is hidden.
pd.set_option("display.max_colwidth", None)
for _, failure in results[~results["passed"]].iterrows():
    print(f"QUESTION: {failure['question']}\nEXPECTED: {failure['expected']}\nGOT: {failure['answer']}\n")
results[["test", "question", "passed"]]

,test,question,passed
0,number,How many stops were there in 2024?,True
1,number,how many terry stops happened in 2019,True
2,number,How many stops ended in arrest in 2023?,True
3,number,What is the overall arrest rate?,True
4,number,arest rate in 2020?,True
5,number,What percent of stops in 2022 involved a frisk?,True
6,number,How often are women frisked?,True
7,number,How often was a weapon found in the North precinct?,True
8,number,What share of shoplifting calls end in arrest?,True
9,number,how often do robbery calls lead to an arrest,True
